# ClipForge Personal
Paste your own YouTube video or replay URL, set clip count and length, then run cells top-to-bottom. Only clip videos you have permission to use. Live streams must have replay/recording.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install yt-dlp faster-whisper

In [ ]:
# Settings
YOUTUBE_URL = "" # Paste video/replay URL
NUMBER_OF_CLIPS = 10
CLIP_SECONDS = 35

In [ ]:
import os, subprocess, shutil, re, zipfile
from pathlib import Path
assert YOUTUBE_URL.startswith(("https://youtube.com/","https://www.youtube.com/","https://youtu.be/","https://m.youtube.com/")), "Add a YouTube URL in Settings."
VIDEO="/content/source.mp4"
subprocess.run(["yt-dlp","-f","bestvideo[height<=1080]+bestaudio/best[height<=1080]","--merge-output-format","mp4","-o",VIDEO,YOUTUBE_URL],check=True)

In [ ]:
from faster_whisper import WhisperModel
gpu=shutil.which("nvidia-smi") is not None
model=WhisperModel("small",device="cuda" if gpu else "cpu",compute_type="float16" if gpu else "int8")
segments, info=model.transcribe(VIDEO,vad_filter=True)
transcript=[{"start":float(s.start),"text":s.text.strip()} for s in segments]
def score(t):
    words=re.findall(r"\\w+",t.lower())
    hooks=["but","however","secret","why","how","mistake","never","always","truth","actually","problem","solution","best","crazy","wait"]
    return min(25,len(words))*1.5+sum(3 for h in hooks if h in words)+min(12,len(t)//35)
cand=[]
for s in transcript:
    start=max(0,s["start"]-CLIP_SECONDS*.25)
    cand.append((score(s["text"]),start,s["text"]))
cand.sort(reverse=True)
picked=[]
for c in cand:
    if all(abs(c[1]-p[1])>CLIP_SECONDS*.65 for p in picked): picked.append(c)
    if len(picked)>=NUMBER_OF_CLIPS: break
picked.sort(key=lambda x:x[1])
print("Highlights:",len(picked))

In [ ]:
outdir="/content/clipforge_clips"
shutil.rmtree(outdir,ignore_errors=True); os.makedirs(outdir,exist_ok=True)
clips=[]
for i,(sc,start,txt) in enumerate(picked,1):
    dest=f"{outdir}/clip_{i:02d}.mp4"
    cmd=["ffmpeg","-y","-ss",str(start),"-i",VIDEO,"-t",str(CLIP_SECONDS),"-vf","scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920","-c:v","libx264","-preset","veryfast","-crf","23","-c:a","aac","-b:a","128k",dest]
    subprocess.run(cmd,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    clips.append(dest)
    print("Done:",dest)
zip_path="/content/ClipForge_personal_clips.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in clips: z.write(p,arcname=Path(p).name)
from google.colab import files
files.download(zip_path)

Moment selection is transcript keyword/length scoring, not guaranteed viral AI. Free Colab sessions can disconnect on long videos.